In [94]:
%load_ext autoreload
%autoreload 2
# imports
import torch
from dataset_utils import *
from ClassificationHeads import *
from transformers import DistilBertTokenizer, AutoModel
from pipeline_utils import create_profile_vector, create_tweet_vectors, train_classifier, test_classifier, \
    multi_svm_training_loop, confidence_classifier_training_loop
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
from sklearn.preprocessing import RobustScaler
from sklearn.datasets import make_blobs
import numpy as np
import matplotlib.pyplot as plt
import umap

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Testing binary SVM Implementation on dummy data

# create dummy data
x, y = make_blobs(n_samples=200, centers=2, random_state=42, cluster_std=1.5)
y = list(map(lambda i: -1 if i == 0 else 1, y))
plt.scatter(x[:, 0], x[:, 1], c=y)
plt.title('Ground Truth')
plt.show()


svm_classifier = SVM(2)
x = torch.tensor(x).float()
y = torch.tensor(y).float()
# train loop
C = 1.0
learning_rate = 0.01
optimizer = torch.optim.SGD(svm_classifier.parameters(), lr=learning_rate)

for epoch in range(5):
    optimizer.zero_grad()

    outputs, predictions = svm_classifier(x)

    # calculate loss
    regularization_loss = 0.5 * torch.sum(svm_classifier.get_weights() ** 2)
    hinge_loss = torch.mean(torch.clamp(1 - y * outputs, min=0))
    loss = regularization_loss + C * hinge_loss

    # backwards pass & optimization
    loss.backward()
    optimizer.step()

# plot
plt.scatter(x[:, 0], x[:, 1], c=predictions.detach().numpy())
plt.title('Predictions')
plt.show()


In [ ]:
# testing multi-label SVM Implementation on dummy data

# create dummy data
x, y = make_blobs(n_samples=200, centers=5, cluster_std=1.5)
plt.scatter(x[:, 0], x[:, 1], c=y)
plt.title('Ground Truth')
plt.show()

multi_svm_classifier = OneAgainstRestSVM(2,5)
x = torch.tensor(x).float()
y = torch.tensor(y).int()

# encode labels into (-1,-1,-1,...,1,...,-1)
remapped_y = torch.full((len(y),5),-1)
remapped_y[torch.arange(len(y)), y] = 1

# train loop
C = 1.0
learning_rate = 0.1
optimizer = torch.optim.SGD(multi_svm_classifier.parameters(), lr=learning_rate)

for epoch in range(5):
    optimizer.zero_grad()

    outputs = multi_svm_classifier(x)
    predictions = outputs.argmax(1)
    # calculate loss
    regularization_loss = 0.5 * torch.sum(multi_svm_classifier.get_weights() ** 2)
    hinge_loss = torch.mean(torch.clamp(1 - remapped_y * outputs, min=0))
    loss = regularization_loss + C * hinge_loss

    # backwards pass & optimization
    loss.backward()
    optimizer.step()

# plot
plt.scatter(x[:, 0], x[:, 1], c=predictions.detach().numpy())
plt.title('Predictions')
plt.show()

In [64]:
# testing confidence classifier:

# datasets
dataset_mode = "train"
user_dataset = Cresci17(Cresci17SetTypes.GENUINE_USER, dataset_mode, root="./datasets", custom_label=0)
fake_dataset = Cresci17(Cresci17SetTypes.FAKE_FOLLOWER, dataset_mode, root="./datasets", custom_label=1)
social1_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_1, dataset_mode, root="./datasets", custom_label=1)
social2_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_2, dataset_mode, root="./datasets", custom_label=1)
social3_dataset = Cresci17(Cresci17SetTypes.SOCIAL_SPAM_3, dataset_mode, root="./datasets", custom_label=1)
traditional1_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_1, dataset_mode, root="./datasets", custom_label=1)
traditional2_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_2, dataset_mode, root="./datasets", custom_label=1)
traditional3_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_3, dataset_mode, root="./datasets", custom_label=1)
traditional4_dataset = Cresci17(Cresci17SetTypes.TRADITIONAL_SPAM_4, dataset_mode, root="./datasets", custom_label=1)
combo_train_dataset = InterleavedIterableDataset([user_dataset, fake_dataset, social1_dataset, social2_dataset, social3_dataset,
        traditional1_dataset, traditional2_dataset, traditional3_dataset, traditional4_dataset], "Random")

replay_buffer = ReplayBuffer(50,0)
classifier = OneAgainstRestSVM(23,9).to("cuda")

# -- initial setup
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased").to("cuda")
scaler = RobustScaler(with_centering=False)
dim_reducer = umap.UMAP(n_components=16, n_neighbors=20, min_dist=0.1, metric="cosine", target_metric="categorical", target_weight=0.25)
# -- training loop
ground_truth_labels = []
profile_vectors = []
tweet_vectors = []
for i, sample in enumerate(combo_train_dataset):
    # get feature vectors
    profile_embed = create_profile_vector(sample.user_data)
    tweet_embeds = create_tweet_vectors(sample.tweet_data, tokenizer, model, max_tweets= 50, batch_size = 50)
    # pool tweet vectors
    tweet_vec = torch.mean(tweet_embeds, dim=0, dtype=torch.float32)

    # append feature vectors and ground truth
    profile_vectors.append(profile_embed)
    tweet_vectors.append(tweet_vec)
    ground_truth_labels.append(sample.label)
    print(f"\rIteration: {i}", end="")

# reduce tweet vectors in dimensionality
reduced_tweet_vectors = dim_reducer.fit_transform(X=tweet_vectors, y=torch.tensor(ground_truth_labels))
reduced_tweet_vectors = np.nan_to_num(reduced_tweet_vectors, nan=0.0, posinf=0.0, neginf=0.0)
# scale profile vectors
scaled_profile_vectors = scaler.fit_transform(profile_vectors)
# combine feature vectors
embedding_features = [np.concatenate((t1, t2), axis=0) for t1, t2 in zip(reduced_tweet_vectors, scaled_profile_vectors)]




Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9858.74it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Iteration: 10144

In [61]:
replay_buffer = ReplayBuffer(50,0)
classifier = OneAgainstRestSVM(23,9).to("cuda")
ground_truths, predictions = multi_svm_training_loop(classifier, replay_buffer, embedding_features, ground_truth_labels)

Epoch [1/15] - Loss: 0.7364
Epoch [2/15] - Loss: 0.6296
Epoch [3/15] - Loss: 0.6465
Epoch [4/15] - Loss: 0.6644
Epoch [5/15] - Loss: 0.6544
Epoch [6/15] - Loss: 0.6638
Epoch [7/15] - Loss: 0.6641
Epoch [8/15] - Loss: 0.6611
Epoch [9/15] - Loss: 0.6562
Epoch [10/15] - Loss: 0.6559
Epoch [11/15] - Loss: 0.6621
Epoch [12/15] - Loss: 0.6523
Epoch [13/15] - Loss: 0.6495
Epoch [14/15] - Loss: 0.6418
Epoch [15/15] - Loss: 0.6408


In [62]:

print("[ -- validation results -- ]")
print("accuracy: ", accuracy_score(ground_truths, predictions))
print("recall: ", recall_score(ground_truths, predictions, average="macro"))
print("f1 score: ", f1_score(ground_truths, predictions, average="macro"))
print(confusion_matrix(ground_truths, predictions))

[ -- validation results -- ]
accuracy:  0.6044921875
recall:  0.37270899317418776
f1 score:  0.36397220745710224
[[1428   93   45  163   74   62  160  225  329]
 [ 151 1979   45   38   55   15   47   16   18]
 [   2   45   21    1    0    4    0    0    0]
 [ 166   39    0 2201   41  199   59    0   75]
 [  18   99    1  169   55   13    0    0    1]
 [  82  162   39  199   56  215    3   21   22]
 [   4    2    0    3    0    0    1    5    9]
 [ 103   15    6   30    0    8   49   57   75]
 [ 287   45   13  104    0   22   91  127  233]]


In [95]:
# testing confidence classifier
classifier = ConfidenceClassifier(23,2).to("cuda")
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.01)
ground_truths, predictions = confidence_classifier_training_loop(classifier, criterion, optimizer,embedding_features, ground_truth_labels, bot_label=1)

print("[ -- validation results -- ]")
print("accuracy: ", accuracy_score(ground_truths, predictions))
print("recall: ", recall_score(ground_truths, predictions, average="macro"))
print("f1 score: ", f1_score(ground_truths, predictions, average="macro"))
print(confusion_matrix(ground_truths, predictions))

Epoch [1/15] - Loss: 0.3461
Epoch [2/15] - Loss: 0.2972
Epoch [3/15] - Loss: 0.3051
Epoch [4/15] - Loss: 0.2883
Epoch [5/15] - Loss: 0.2890
Epoch [6/15] - Loss: 0.2762
Epoch [7/15] - Loss: 0.2843
Epoch [8/15] - Loss: 0.2706
Epoch [9/15] - Loss: 0.2879
Epoch [10/15] - Loss: 0.2837
Epoch [11/15] - Loss: 0.2754
Epoch [12/15] - Loss: 0.2711
Epoch [13/15] - Loss: 0.2811
Epoch [14/15] - Loss: 0.2759
Epoch [15/15] - Loss: 0.2834
[ -- validation results -- ]
accuracy:  0.865382932166302
recall:  0.8251738650463301
f1 score:  0.8133805656011177
[[1928  635]
 [ 903 7959]]
